In [1]:
%config Completer.use_jedi = False
%load_ext autoreload

# Imports and Loads

In [4]:
from bs4 import BeautifulSoup
import os
import requests
from zipfile import ZipFile
from io import BytesIO
from datetime import datetime
import pandas as pd
import logging
import csv
from pathlib import Path
from time import sleep

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from dotenv import load_dotenv, find_dotenv

In [5]:
# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    handlers=[logging.FileHandler("../../logs/get_ranks.log"), logging.StreamHandler()],
)
logger = logging.getLogger(__name__)

# Functions

In [6]:
def get_driver_and_cookies():
    """
    Initialize Selenium WebDriver and authenticate with BoardGameGeek.
    Returns authentication cookies needed for subsequent API requests.

    Returns:
        dict: Authentication cookies for BGG API requests
    """
    logger.info("Initializing web driver and getting cookies")
    LOGIN_USERNAME_FIELD = '//*[@id="inputUsername"]'
    LOGIN_PASSWORD_FIELD = '//*[@id="inputPassword"]'
    LOGIN_BUTTON = '//*[@id="mainbody"]/div/div/gg-login-page/div[1]/div/gg-login-form/form/fieldset/div[3]/button[1]'

    load_dotenv(find_dotenv())
    USERNAME = os.getenv("BGG_USERNAME")
    PASSWORD = os.getenv("BGG_PASSWORD")
    if not USERNAME or not PASSWORD:
        logger.error("Missing BGG_USERNAME or BGG_PASSWORD in environment")
        raise ValueError("Missing BGG_USERNAME or BGG_PASSWORD in environment")

    chrome_options = Options()
    chrome_options.add_argument("--headless")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    cookies = {}

    driver = webdriver.Chrome(
        service=Service("/usr/lib/chromium-browser/chromedriver"),
        options=chrome_options,
    )
    logger.info("Chrome driver initialized successfully")

    driver.get("https://boardgamegeek.com/login")
    login = WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.XPATH, LOGIN_USERNAME_FIELD))
    )
    password = WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.XPATH, LOGIN_PASSWORD_FIELD))
    )

    login_button = WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.XPATH, LOGIN_BUTTON))
    )

    login.send_keys(USERNAME)
    password.send_keys(PASSWORD)

    login_button.click()
    sleep(1)
    logger.info("Successfully logged in to BoardGameGeek")

    selenium_cookies = driver.get_cookies()
    for cookie in selenium_cookies:
        cookies[cookie["name"]] = cookie["value"]
    logger.info("Successfully retrieved cookies")
    return cookies

# Code

In [7]:
cookies = get_driver_and_cookies()

2025-09-10 14:55:22,417 - __main__ - INFO - Initializing web driver and getting cookies
2025-09-10 14:55:23,241 - __main__ - INFO - Chrome driver initialized successfully
2025-09-10 14:55:25,535 - __main__ - INFO - Successfully logged in to BoardGameGeek
2025-09-10 14:55:26,189 - __main__ - INFO - Successfully retrieved cookies


In [8]:
logger.info("Fetching boardgame ranks")
bg_ranks_pg_url = "https://boardgamegeek.com/data_dumps/bg_ranks"
resp = requests.get(bg_ranks_pg_url, cookies=cookies)
soup = BeautifulSoup(resp.content, "html.parser")
bg_ranks_url = soup.find("div", {"id": "maincontent"})("a")[0]["href"]
bg_ranks_zip = requests.get(bg_ranks_url)
queried_at_utc = datetime.now().replace(microsecond=0).isoformat()

2025-09-10 14:55:44,600 - __main__ - INFO - Fetching boardgame ranks


In [15]:
save_file = True
with ZipFile(BytesIO(bg_ranks_zip.content)) as archive:
    with archive.open("boardgames_ranks.csv") as csv_file:
        df_bg_ranks = pd.read_csv(csv_file)
        df_bg_ranks["name"] = df_bg_ranks["name"].str.replace("[“”]", '"', regex=True)
        df_bg_ranks["queried_at_utc"] = queried_at_utc
        logger.info(f"Successfully loaded {len(df_bg_ranks)} boardgames")
        if save_file:
            # Create data directory if it doesn't exist
            data_dir = Path(__file__).parent.parent.parent / "data" / "crawler"
            data_dir.mkdir(parents=True, exist_ok=True)
            
            output_file = data_dir / f"boardgame_ranks_{queried_at_utc[:10].replace('-','')}.csv"
            df_bg_ranks.to_csv(
                output_file,
                index=False,
                sep="|",
                escapechar="\\",
                quoting=csv.QUOTE_NONE,
            )
            logger.info(f"Saved rankings to {output_file}")

2025-09-10 14:59:30,320 - __main__ - INFO - Successfully loaded 168655 boardgames


NameError: name '__file__' is not defined

In [14]:
df_bg_ranks.head()

,id,name,yearpublished,rank,bayesaverage,average,usersrated,is_expansion,abstracts_rank,cgs_rank,childrensgames_rank,familygames_rank,partygames_rank,strategygames_rank,thematic_rank,wargames_rank,queried_at_utc
0,224517,Brass: Birmingham,2018,1,8.40111,8.57497,53908,0,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,2025-09-10T14:55:45
1,161936,Pandemic Legacy: Season 1,2015,2,8.35706,8.51076,56170,0,NaN,NaN,NaN,NaN,NaN,3.0,1.0,NaN,2025-09-10T14:55:45
2,342942,Ark Nova,2021,3,8.35247,8.54017,54993,0,NaN,NaN,NaN,NaN,NaN,2.0,NaN,NaN,2025-09-10T14:55:45
3,174430,Gloomhaven,2017,4,8.31792,8.55649,65564,0,NaN,NaN,NaN,NaN,NaN,4.0,2.0,NaN,2025-09-10T14:55:45
4,316554,Dune: Imperium,2020,5,8.22478,8.41712,54270,0,NaN,NaN,NaN,NaN,NaN,7.0,NaN,NaN,2025-09-10T14:55:45
